# Split-SLM Two-Beam Optimization

Optimize two spatially separated beams independently by splitting the SLM into left and right halves.

## 1. Imports and Parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from initial_holograms import curvature_hologram
from loss_functions import build_circular_mask
from optimizer import CGOptimizer
from optical_planes import CameraPlane, SLMPlane
from propagator import Propagator
from target_profiles import apply_psf_smoothing, build_rectangle_target

# Native SLM and compute grid.
full_slm_shape = (1080, 1920)
superpixel_size = 4
slm_shape = (full_slm_shape[0] // superpixel_size, full_slm_shape[1] // superpixel_size)
half_slm_shape = (slm_shape[0], slm_shape[1] // 2)
padding_factor = 2
half_camera_shape = (half_slm_shape[0] * padding_factor, half_slm_shape[1] * padding_factor)

# Optical parameters.
slm_pixel_pitch_um = 8.0
camera_pixel_pitch_um = 3.45
focal_length_mm = 200.0
wavelength_left_nm = 420.0
wavelength_right_nm = 456.0

# slmsuite wavefront-calibration files.
left_calibration_h5_path = "10806-SLM-wavefront_superpixel-calibration_00036.h5"
right_calibration_h5_path = "10806-SLM-wavefront_superpixel-calibration_00030.h5"
slmsuite_camera_shape = full_slm_shape
slmsuite_wavefront_r2_threshold = 0.5
slmsuite_remove_background = True
slmsuite_apply_calibration = True
slmsuite_amplitude_key = "amplitude"
slmsuite_phase_key = "phase"

# Target: same vertical rectangle at the center of each half-SLM focal plane.
vertical_target_width_x_um = 30.0
vertical_target_height_y_um = 300.0
psf_sigma_x_um = 10.0
psf_sigma_y_um = 10.0
mask_radius_margin_um = 100.0

# Initial hologram parameters for each half.
initial_phase_linear_tilt = 0.0
initial_phase_astigmatism_weight = 0.0
initial_phase_quadratic_curvature = 3.6e-3
initial_phase_linear_angle_rad = np.pi / 4.0
initial_phase_conical_weight = 0.0

# CG parameters.
optimizer_maxiter = 100
loss_scale = 1e12
optimize_phase = True


## 2. Load Inputs, Split SLM, and Build Two Optimizers

In [ ]:
def downsample_by_superpixel(data, superpixel_size):
    data_array = np.asarray(data, dtype=float)
    if data_array.ndim != 2:
        raise ValueError(f"data must be 2D, got shape {data_array.shape}.")
    if data_array.shape[0] % superpixel_size != 0 or data_array.shape[1] % superpixel_size != 0:
        raise ValueError(f"data shape must be divisible by superpixel_size, got data_shape={data_array.shape}, superpixel_size={superpixel_size}.")
    ny = data_array.shape[0] // superpixel_size
    nx = data_array.shape[1] // superpixel_size
    return data_array.reshape(ny, superpixel_size, nx, superpixel_size).mean(axis=(1, 3))


def downsample_phase_by_superpixel(phase, superpixel_size):
    phase_array = np.asarray(phase, dtype=float)
    if phase_array.ndim != 2:
        raise ValueError(f"phase must be 2D, got shape {phase_array.shape}.")
    real_part = downsample_by_superpixel(np.cos(phase_array), superpixel_size)
    imag_part = downsample_by_superpixel(np.sin(phase_array), superpixel_size)
    return np.angle(real_part + 1j * imag_part)


def require_calibration_array(calibration_results, key, expected_shape):
    if key not in calibration_results:
        available_keys = sorted(str(item) for item in calibration_results.keys())
        raise KeyError(f"Calibration result key {key!r} was not found. Available keys: {available_keys}.")
    array = np.asarray(calibration_results[key], dtype=float)
    if array.shape != expected_shape:
        raise ValueError(f"Calibration result {key!r} shape must be {expected_shape}, got {array.shape}.")
    if not np.all(np.isfinite(array)):
        raise ValueError(f"Calibration result {key!r} contains NaN or infinite values.")
    return array


def load_slmsuite_input_beam_and_phase(calibration_h5_path, full_slm_shape, superpixel_size, slm_pixel_pitch_um, wavelength_nm, camera_shape, r2_threshold, remove_background, apply_calibration, amplitude_key, phase_key):
    from slmsuite.hardware.cameras.simulated import SimulatedCamera
    from slmsuite.hardware.cameraslms import FourierSLM
    from slmsuite.hardware.slms.simulated import SimulatedSLM

    slm_size_xy = (int(full_slm_shape[1]), int(full_slm_shape[0]))
    camera_size_xy = (int(camera_shape[1]), int(camera_shape[0]))
    slm = SimulatedSLM(slm_size_xy, pitch_um=(slm_pixel_pitch_um, slm_pixel_pitch_um), wav_um=wavelength_nm / 1000.0)
    camera = SimulatedCamera(slm, resolution=camera_size_xy)
    fourier_slm = FourierSLM(camera, slm)
    fourier_slm.load_calibration("wavefront_superpixel", file_path=calibration_h5_path)
    calibration_results = fourier_slm.wavefront_calibration_superpixel_process(
        plot=False,
        r2_threshold=r2_threshold,
        remove_background=remove_background,
        apply=apply_calibration,
    )
    amplitude_full = require_calibration_array(calibration_results, amplitude_key, full_slm_shape)
    phase_full = require_calibration_array(calibration_results, phase_key, full_slm_shape)
    amplitude = downsample_by_superpixel(np.clip(amplitude_full, 0.0, None), superpixel_size)
    phase = downsample_phase_by_superpixel(phase_full, superpixel_size)
    if np.max(amplitude) <= 0:
        raise ValueError("Loaded slmsuite amplitude contains no positive values after downsampling.")
    return amplitude / np.max(amplitude), np.mod(phase, 2.0 * np.pi), calibration_results


def crop_half_slm(amplitude, phase, side):
    if side not in ("left", "right"):
        raise ValueError(f"side must be 'left' or 'right', got {side}.")
    amplitude_array = np.asarray(amplitude, dtype=float)
    phase_array = np.asarray(phase, dtype=float)
    if amplitude_array.shape != slm_shape or phase_array.shape != slm_shape:
        raise ValueError(f"amplitude and phase must both have shape {slm_shape}, got {amplitude_array.shape} and {phase_array.shape}.")
    if slm_shape[1] % 2 != 0:
        raise ValueError(f"SLM x size must be even for a left/right split, got {slm_shape[1]}.")
    split_index = slm_shape[1] // 2
    if side == "left":
        amplitude_half = amplitude_array[:, :split_index]
        phase_half = phase_array[:, :split_index]
    else:
        amplitude_half = amplitude_array[:, split_index:]
        phase_half = phase_array[:, split_index:]
    if np.max(amplitude_half) <= 0:
        raise ValueError(f"{side} half input amplitude contains no positive values.")
    return amplitude_half / np.max(amplitude_half), phase_half.copy()


def build_half_optimizer(wavelength_nm, input_amplitude, input_phase, label):
    initial_hologram = curvature_hologram(
        shape=half_slm_shape,
        linear_tilt=initial_phase_linear_tilt,
        astigmatism_weight=initial_phase_astigmatism_weight,
        quadratic_curvature=initial_phase_quadratic_curvature,
        linear_angle_rad=initial_phase_linear_angle_rad,
        conical_weight=initial_phase_conical_weight,
    )
    slm = SLMPlane(half_slm_shape, wavelength_nm, slm_pixel_pitch_um, superpixel_size, input_amplitude, input_phase, initial_hologram)
    camera = CameraPlane(half_camera_shape, wavelength_nm, (1.0, 1.0), camera_pixel_pitch_um, np.zeros(half_camera_shape), np.zeros(half_camera_shape))
    propagator = Propagator(slm, camera, focal_length_mm, padding_factor)
    ideal_target = build_rectangle_target(
        half_camera_shape,
        propagator.camera_plane.x_axis_um,
        propagator.camera_plane.y_axis_um,
        vertical_target_width_x_um,
        vertical_target_height_y_um,
    )
    target_amplitude = apply_psf_smoothing(ideal_target, psf_sigma_x_um, psf_sigma_y_um, propagator.camera_plane.scale_um)
    target_phase = np.zeros_like(target_amplitude)
    mask_center_x_um = 0.5 * (propagator.camera_plane.x_axis_um[0] + propagator.camera_plane.x_axis_um[-1])
    mask_center_y_um = 0.5 * (propagator.camera_plane.y_axis_um[0] + propagator.camera_plane.y_axis_um[-1])
    mask_radius_um = max(vertical_target_width_x_um, vertical_target_height_y_um) / 2.0 + mask_radius_margin_um
    target_mask = build_circular_mask(
        half_camera_shape,
        propagator.camera_plane.x_axis_um,
        propagator.camera_plane.y_axis_um,
        mask_center_x_um,
        mask_center_y_um,
        mask_radius_um,
    )
    target_plane = CameraPlane(half_camera_shape, wavelength_nm, propagator.camera_plane.scale_um, camera_pixel_pitch_um, target_amplitude, target_phase)
    optimizer = CGOptimizer(propagator, target_plane, target_mask)
    optimizer.set_initial_hologram_array(initial_hologram)
    return optimizer, target_plane, target_mask, initial_hologram


left_full_amplitude, left_full_phase, left_calibration_results = load_slmsuite_input_beam_and_phase(
    left_calibration_h5_path,
    full_slm_shape,
    superpixel_size,
    slm_pixel_pitch_um,
    wavelength_left_nm,
    slmsuite_camera_shape,
    slmsuite_wavefront_r2_threshold,
    slmsuite_remove_background,
    slmsuite_apply_calibration,
    slmsuite_amplitude_key,
    slmsuite_phase_key,
)
right_full_amplitude, right_full_phase, right_calibration_results = load_slmsuite_input_beam_and_phase(
    right_calibration_h5_path,
    full_slm_shape,
    superpixel_size,
    slm_pixel_pitch_um,
    wavelength_right_nm,
    slmsuite_camera_shape,
    slmsuite_wavefront_r2_threshold,
    slmsuite_remove_background,
    slmsuite_apply_calibration,
    slmsuite_amplitude_key,
    slmsuite_phase_key,
)

left_input_amplitude, left_input_phase = crop_half_slm(left_full_amplitude, left_full_phase, "left")
right_input_amplitude, right_input_phase = crop_half_slm(right_full_amplitude, right_full_phase, "right")
left_optimizer, left_target_plane, left_target_mask, left_initial_hologram = build_half_optimizer(wavelength_left_nm, left_input_amplitude, left_input_phase, "left")
right_optimizer, right_target_plane, right_target_mask, right_initial_hologram = build_half_optimizer(wavelength_right_nm, right_input_amplitude, right_input_phase, "right")

print("Full superpixel SLM shape:", slm_shape)
print("Half superpixel SLM shape:", half_slm_shape)
print("Half camera/optimization shape:", half_camera_shape)
print("Left input shape:", left_input_amplitude.shape)
print("Right input shape:", right_input_amplitude.shape)

fig, axes = plt.subplots(2, 3, figsize=(13.5, 7.6), constrained_layout=True)
axes[0, 0].imshow(left_input_amplitude**2, origin="lower", cmap="magma", aspect="equal")
axes[0, 0].set_title("420 nm left-half input intensity")
axes[0, 1].imshow(left_input_phase, origin="lower", cmap="twilight", aspect="equal")
axes[0, 1].set_title("420 nm left-half input phase")
axes[0, 2].imshow(left_target_plane.amplitude**2 * left_target_mask, origin="lower", cmap="inferno", aspect="equal")
axes[0, 2].set_title("420 nm centered vertical target")
axes[1, 0].imshow(right_input_amplitude**2, origin="lower", cmap="magma", aspect="equal")
axes[1, 0].set_title("456 nm right-half input intensity")
axes[1, 1].imshow(right_input_phase, origin="lower", cmap="twilight", aspect="equal")
axes[1, 1].set_title("456 nm right-half input phase")
axes[1, 2].imshow(right_target_plane.amplitude**2 * right_target_mask, origin="lower", cmap="inferno", aspect="equal")
axes[1, 2].set_title("456 nm centered vertical target")
plt.show()


## 3. Optimize Both Halves

In [ ]:
left_optimizer.optimize(
    maxiter=optimizer_maxiter,
    loss_scale=loss_scale,
    optimize_phase=optimize_phase,
)
right_optimizer.optimize(
    maxiter=optimizer_maxiter,
    loss_scale=loss_scale,
    optimize_phase=optimize_phase,
)

print("Left-half optimization finished")
print("Left loss evaluations:", len(left_optimizer.loss_history))
print("Left accepted CG iterations:", len(left_optimizer.iteration_loss_history))
print("Right-half optimization finished")
print("Right loss evaluations:", len(right_optimizer.loss_history))
print("Right accepted CG iterations:", len(right_optimizer.iteration_loss_history))


## 4. Show Results and Combine Holograms

In [ ]:
left_result = left_optimizer.get_result_summary()
right_result = right_optimizer.get_result_summary()
left_hologram = left_optimizer.get_final_hologram()
right_hologram = right_optimizer.get_final_hologram()
combined_superpixel_hologram = np.mod(np.concatenate([left_hologram, right_hologram], axis=1), 2.0 * np.pi)
combined_full_resolution_hologram = np.repeat(np.repeat(combined_superpixel_hologram, superpixel_size, axis=0), superpixel_size, axis=1)

print("Left efficiency:", left_result.efficiency)
print("Left fidelity:", left_result.fidelity)
print("Left RMS error:", left_result.rms_error)
print("Left phase error:", left_result.phase_error)
print("Right efficiency:", right_result.efficiency)
print("Right fidelity:", right_result.fidelity)
print("Right RMS error:", right_result.rms_error)
print("Right phase error:", right_result.phase_error)
print("Combined superpixel hologram shape:", combined_superpixel_hologram.shape)
print("Combined full-resolution hologram shape:", combined_full_resolution_hologram.shape)

left_optimizer.plot_loss_history()
plt.show()
right_optimizer.plot_loss_history()
plt.show()
left_optimizer.plot_result_summary()
plt.show()
right_optimizer.plot_result_summary()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.2), constrained_layout=True)
axes[0].imshow(left_hologram, origin="lower", cmap="twilight", aspect="equal")
axes[0].set_title("Left-half hologram")
axes[1].imshow(right_hologram, origin="lower", cmap="twilight", aspect="equal")
axes[1].set_title("Right-half hologram")
axes[2].imshow(combined_superpixel_hologram, origin="lower", cmap="twilight", aspect="equal")
axes[2].set_title("Combined superpixel hologram")
plt.show()
